**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Diffusion Models

The generative method behind modern image synthesis, told as a *signal processing* story: corrupt data with Gaussian noise step by step, train a network to **denoise**, then run the corruption in reverse. We train a complete diffusion model on 2-D data in minutes and watch noise crystallize into structure.

## 1. Pre-requisites

- [Intro to PyTorch](../Intro_DL_4_Physics/intro_pytorch/intro_pytorch.ipynb).
- [Random Variables](../Intro_Math/Analysis/Random_Variables.ipynb) (Gaussians compose).
- [Representation Learning](./Representation_Learning.ipynb) S2 for the generative-model context.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(0); rng = np.random.default_rng(0)

# target distribution: two moons — structured, multimodal, low-dimensional
def moons(n):
    t = rng.uniform(0, np.pi, n)
    top = np.stack([np.cos(t), np.sin(t)], 1)
    bot = np.stack([1 - np.cos(t), 0.4 - np.sin(t)], 1)
    X = np.concatenate([top[:n//2], bot[n//2:]]) + 0.06*rng.standard_normal((n, 2))
    return ((X - X.mean(0)) / X.std(0)).astype(np.float32)

X = torch.from_numpy(moons(6000))
plt.figure(figsize=(3.6, 3.2)); plt.scatter(*X.T, s=2, alpha=0.3)
plt.title("the distribution we want to SAMPLE from"); plt.axis("equal")
plt.tight_layout(); plt.show()

**What just happened.** Six thousand points forming two interleaved crescents, standardized to zero mean and unit variance. This is the target: by the end of the notebook a model will produce *fresh* points from this shape, having started from nothing but Gaussian noise.

**Note what makes this a real generative problem rather than a toy.** The distribution is **multimodal** — two separated arcs, so no unimodal model can represent it — and it lies on a **curved, thin manifold** in 2-D, so the support is a measure-zero set that a naive density fit will smear across. Both properties are exactly what break simple approaches, and both are what generative modelling exists to handle.

**Be clear about what "generate" means here, because it is a stronger requirement than it sounds.** We are not asking for a function that classifies these points, or scores them, or compresses them. We are asking to **draw new samples** from a distribution we only ever see through 6,000 examples. That is a harder problem than any supervised task in the ML track so far, and the reason diffusion's central move — reduce it to a regression — is worth the whole workshop.

**The standardization is not cosmetic.** The forward process is defined in terms of a noise schedule with fixed variances, so the data must be on a comparable scale for $\sqrt{\bar\alpha_t}x_0$ and $\sqrt{1-\bar\alpha_t}\varepsilon$ to trade off sensibly. Feed in data with variance 100 and the "noise" is negligible until very late in the chain; feed in variance 0.01 and the signal is destroyed in a handful of steps. **The schedule assumes unit-scale data**, which is why every diffusion pipeline normalises first.

**Two dimensions is a deliberate and consequential choice.** It means the whole distribution fits in one scatter plot, so you can *see* whether the model succeeded — and, more importantly, it means the audit in Session 2 can compute honest distributional statistics. In image space no such check exists; FID is a crude proxy and human evaluation is still the standard. **Low dimension is what lets this notebook grade itself**, and that is worth more than realism at this stage.

**Keep this picture in view for the rest of the workshop.** In Session 1 it dissolves into noise; in Session 2 noise crystallises back into it. The final scatter should be indistinguishable from this one — and the cell after it checks whether "indistinguishable to the eye" survives contact with actual numbers.

---
### 🕐 Session 1 of 2 — *The Forward Process & the Denoising Objective* (~35 min)
**Goal:** destroy data with scheduled noise; train a network to predict the noise.
**Feeds into:** Session 2 (sampling = reverse diffusion).

---

## 2. Destruction Is Easy — Learn to Undo It

💡 **Intuition.** Generating from scratch is hard; *removing a little noise* is easy — it's [Wiener denoising's](../Intro_DSP/Statistical_Signal_Processing.ipynb) cousin, a regression problem. Diffusion's insight: chain the easy problem. Define a forward process that gradually noises data into pure Gaussian ($x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\varepsilon$ — Gaussians compose, so any step is one formula), and train one network $\varepsilon_\theta(x_t, t)$ to **predict the noise** that was added. Denoising at *every* noise level = knowing the path from chaos back to data.

In [ ]:
# visualize the forward death of the data

# YOUR CODE HERE


**What just happened.** Four snapshots of the same 6,000 points being destroyed. At $t = 0$ the two moons are crisp. By $t = 60$ they are smeared but still recognisably two arcs. At $t = 120$ only a vague elongation survives. At $t = 199$ it is a featureless Gaussian blob — the structure is gone.

**Read the order of destruction, because it is the whole DSP story.** Fine detail dies first; **coarse structure survives longest**. The gap between the moons persists long after the crispness of each arc has dissolved. That is progressive low-pass-plus-noise: adding white noise raises the floor uniformly, so the low-amplitude high-frequency content disappears under it before the large-scale shape does.

**Which immediately tells you what the reverse process must do.** If detail dies last-in, it is recovered first-out — the reverse chain builds **gross shape early and detail late**. Generation is spectral refinement. And this explains, in one sentence, why the same trained model can denoise, inpaint, and super-resolve: those tasks are *partial* trips along a chain the model already knows end to end.

**Note that no simulation was needed to draw this figure.** Each panel jumps straight to its $t$ via $x_t = \sqrt{\bar\alpha_t}x_0 + \sqrt{1-\bar\alpha_t}\varepsilon$ — one line, because Gaussians compose and a chain of Gaussian steps is a single Gaussian step with accumulated variance. **That closed form is what makes training affordable**: the loop samples a random $t$ per example and jumps there, instead of rolling the chain forward $t$ times. Without it the method costs $O(T)$ per training example and nobody builds it.

**Check the final panel against the schedule, though, because "pure noise" is an approximation here.** With $\beta$ running from $10^{-4}$ to $0.04$ over 200 steps, $\sum\beta_t \approx 4.0$ and $\bar\alpha_T \approx 0.017$ — so $\sqrt{\bar\alpha_T} \approx 0.13$ and about **13% of the original signal amplitude is still present** at $t = 199$. Sampling in Session 2 starts from exact $\mathcal{N}(0, I)$, so the reverse chain begins at a slightly different distribution from where the forward chain ended.

**The mismatch is small and worth knowing about rather than worrying about.** It costs a little sample quality and is invisible at this scale, but it is exactly why production schedules use 1000 steps, or a cosine $\bar\alpha$ profile that drives the signal much closer to zero at the end. **A demonstration that admits its own approximations is more useful than one that asserts "now it is pure noise"** — and you can verify this one directly by printing `abar[-1].sqrt()`.

In [ ]:
# the denoiser: predicts ε from (x_t, t) — t is embedded sinusoidally, like a transformer position

# YOUR CODE HERE


**What just happened.** A complete diffusion model trained in 4,000 steps on a laptop, and the loss reads **0.9189 → 0.3236 → 0.4127 → 0.3509**.

**Those numbers are not monotone, and that is expected rather than a failure.** Each print is a *single* 256-sample minibatch with a randomly drawn $t$, and the achievable loss depends heavily on which $t$ was drawn. At large $t$, $x_t \approx \varepsilon$, so the network can nearly read the answer off its input and the loss is small. At small $t$, $x_t \approx x_0$ and the noise has barely perturbed anything, so $\varepsilon$ is close to unrecoverable and the loss approaches **1.0** — the variance of $\varepsilon$ itself. **The bouncing is $t$-sampling variance, not instability.** A room not warned about this concludes the run diverged.

**Which gives you the right yardstick: 1.0, not 0.** A network that outputs zero achieves MSE 1.0, since $\varepsilon \sim \mathcal{N}(0,I)$. So 0.35 means the model explains about 65% of the noise variance averaged over $t$ — and no model can drive this to zero, because at $t$ near 0 the noise genuinely is not identifiable from the input. **The loss floor is a property of the problem, not of the network.**

**Now look at the objective itself, because its plainness is the point of the workshop.** `((model(xt, t) - eps)**2).mean()` — predict the noise you added. No adversary, no ELBO, no reconstruction term, no KL, no posterior. Anyone who has fought GAN mode collapse or VAE posterior collapse should find this startling: **the training stability of diffusion is most of why it displaced both**. The variational derivation exists and simplifies down to exactly this expression.

**Two implementation details are doing real work.** The **time embedding** — sinusoidal features of $t$, the same encoding transformers use for position — is what lets *one* network serve all 200 noise levels; without it you would need 200 models or a model that cannot tell which regime it is in. And **SiLU** rather than ReLU: the denoiser must output a smooth vector field, and ReLU's kink produces visible artifacts in the sampling trajectory.

**Finally, note the scale of what this is.** 4,000 steps, a 4-layer MLP, two dimensions, under a minute. Stable Diffusion is **the same loss and the same schedule** with a U-Net denoiser, a latent space, text conditioning, and six orders of magnitude more compute. The idea does not get more complicated as it gets bigger — which is precisely why it is worth training one at this scale first.

---
### 🕐 Session 2 of 2 — *Sampling: Running Time Backwards* (~40 min)
**Goal:** start from pure noise and iteratively denoise into fresh samples.
**Builds on:** Session 1.

---

## 3. The Reverse Process

💡 **Intuition.** To sample: start at $x_T \sim \mathcal{N}(0, I)$ and repeatedly apply the learned denoiser, stepping $t = T{-}1, \dots, 0$, re-injecting a *little* fresh noise each step (the stochasticity keeps samples diverse — drop it and you get DDIM's deterministic cousin). Each step is a small, easy denoise; a few hundred of them compound into creation. Image generators are this exact loop with a U-Net denoiser and billions of pixels.

In [ ]:

# YOUR CODE HERE


**What just happened.** Three thousand points of pure Gaussian noise walked backwards through 200 denoising steps and **became two moons**. The snapshots run blob → elongated cloud → two lobes → crisp arcs. Nothing in the model was ever asked to generate anything; it was asked, 200 times, "what would this have been without the noise?"

**Compare the order of appearance against the forward process, because it is a prediction being confirmed.** Session 1 showed detail dying first and coarse structure surviving longest. Run the chain backwards and the reverse must hold: **gross shape first, detail last**. The panels show exactly that — the gap between the moons emerges long before either arc becomes sharp. Generation as spectral refinement, low frequencies to high, and the figure is evidence for it rather than an illustration of it.

**The key line is the inversion, and it is worth reading aloud.** `x0_hat = (x - (1-abar).sqrt()*eps_hat) / abar.sqrt()` is the forward formula solved for $x_0$. The network predicts the noise; algebra converts that into a guess at the clean data; the guess is then re-noised to level $t-1$. **At no point does the model output a sample** — it outputs a noise estimate, and the sampler does the rest.

**The fresh noise added at each step is a deliberate choice with a named alternative.** `betas[t_i].sqrt()*torch.randn_like(x)` keeps the chain stochastic, so nearby starting points can land in genuinely different places. Delete it and the map becomes deterministic — that is DDIM, which is faster and allows far fewer steps, at the cost of less diversity per unit of exploration. Same trained model, different sampler; both are legitimate and the choice is made at sampling time.

**Note the cost asymmetry, since it is the method's defining practical weakness.** Training touched one random $t$ per example. Sampling requires **200 sequential forward passes per sample**, and they cannot be parallelised across $t$ because each depends on the last. That is why diffusion image generation takes seconds where a GAN takes milliseconds, and why an entire literature — DDIM, DPM-Solver, consistency models, distillation — exists purely to cut the step count. The 200 here could probably be 20 with a better sampler.

**One caveat on where the chain starts.** Sampling begins at exact $\mathcal{N}(0, I)$, but the forward process at $t = 199$ left about 13% of the signal amplitude in place ($\sqrt{\bar\alpha_T} \approx 0.13$). So the reverse chain starts from a slightly different distribution than the one the forward chain reached — a small train/sample mismatch, invisible at this scale, and the reason real schedules use more steps or a cosine profile.

**And resist declaring victory from the picture alone.** It looks right, which is not the same as being right — the moons could be subtly the wrong width, one mode could be underpopulated, or the model could be reproducing training points. The next cell exists precisely because "it looks like the data" is not a measurement.

In [ ]:
# quantitative check: do generated samples match the data's statistics?
# and the harder test: fraction of generated points close to the true manifold

# YOUR CODE HERE


**What just happened.** Three independent checks on whether the generated distribution actually matches the data, rather than merely looking like it:

| statistic | data | generated |
|---|---|---|
| mean | $[0.00, -0.00]$ | $[0.04, 0.02]$ |
| variances | $1.00,\ 1.00$ | $1.06,\ 1.02$ |
| correlation | $-0.47$ | $-0.52$ |

and the 90th-percentile nearest-neighbour distance from a generated point to the real data is **0.040** — on unit-variance data, so 4% of a standard deviation. Nine tenths of the generated points sit essentially *on* the manifold.

**The systematic 2–6% variance overshoot is worth noticing rather than rounding away.** Both variances came out above 1.0 and the correlation is slightly stronger than the data's. That is the expected signature of a diffusion sampler that has not fully converged: residual noise from the last few steps has not been removed, so samples are slightly more spread than the target. It is small, it is consistent across both coordinates, and it would shrink with more training steps or a finer schedule.

**Now the important part: what these numbers cannot see.** The nearest-neighbour distance is **minimised by memorisation**. A model that simply reproduced training points would score **0.000** and look perfect by this metric. So 0.040 is evidence the samples are *on* the manifold and no evidence at all that they are *new*. For a generative model, the second question is the one that matters — and this audit does not ask it.

**It is also blind to mode collapse.** Generate only the upper moon and the nearest-neighbour distance stays tiny, while the mean shifts a little and the covariance changes moderately — plausibly within what is reported above. **All three statistics can pass while half the distribution is missing.**

**Both gaps have cheap fixes, and they are the experiment worth running.** For coverage, compute the **reverse** nearest-neighbour distance: for each *training* point, how far is the closest *generated* point? If a mode is missing, that number explodes for the abandoned region while the forward direction stays small. For novelty, compare the forward distance against the typical nearest-neighbour distance *within* the training set — if generated points are systematically closer to training data than training data is to itself, the model is copying.

**Two-moons is 2-D, which is exactly why any of this is checkable.** In image space there is no reliable answer to "does the generated distribution match the data distribution?" — FID is a Gaussian approximation in a feature space chosen for other purposes, and every serious paper still reports human evaluation because the metrics are known to be inadequate. **The audit available here is far stronger than anything available at image scale**, and that is worth knowing before trusting a headline number from a generative-model paper.

**Still, the honest summary is a good one.** One regression loss, one noise schedule, 4,000 training steps on a laptop, and the generated distribution matches the target's first and second moments to a few percent with 90% of samples on the manifold. Swap the MLP for a U-Net and add six orders of magnitude of compute and this is Stable Diffusion — the idea does not get more complicated on the way up.

**The DSP lens, explicitly:** the forward process is progressive low-pass-plus-noise (coarse structure survives longest); the reverse process therefore builds coarse structure first and details last — generation as *spectral refinement*. That's also why diffusion models are natural denoisers, inpainters, and super-resolvers: those are just partial trips along the same chain.

## 4. Conclusion

One regression loss (predict the noise), one schedule, and a walk backwards through it: that's the entire method behind modern generative imagery — and you just trained one.

---
## Where next

- [Representation Learning](./Representation_Learning.ipynb) — VAEs: the previous generation of generation.
- [Statistical Signal Processing](../Intro_DSP/Statistical_Signal_Processing.ipynb) — the denoising theory underneath.
- [Scaling Neural Networks](./Scale_NN/Scale_NN.ipynb) — what it takes to run this at image scale.